# big_randomSVD + rsvd computation with K/4 — recount2 subsamples

Computes SVD using both `bigstatsr::big_randomSVD()` and `rsvd::rsvd()` for the
same subsampled datasets as `07_recount2_coverage_multiplier_imp_fixed.ipynb`,
but with `SVD_K = floor(min(n_samples − 1, n_genes − 1) / 4)`.

- Reads `subsample_info.rds` from each original run directory to recover sample indices.
- Runs `big_randomSVD` on the FBM using `ind.col = sample_idx` with `k = SVD_K`.
- Extracts the submatrix and runs `rsvd::rsvd(Y_sub, k = SVD_K)`.
- Saves results as `svd.rds` (big_randomSVD) and `svd_rsvd.rds` (rsvd) in a new folder:
  `output/recount2_svd_k_div4/c2cp_subsample_{N}_seed_{idx}/`.
- Also copies `subsample_info.rds` and `CLAMP_K.rds` for downstream use in `13_curvature_num_pc.ipynb`.

## Load libraries

In [1]:
start_time <- Sys.time()
cat("big_randomSVD + rsvd (K/4) computation started at:", format(start_time), "\n")

big_randomSVD + rsvd (K/4) computation started at: 2026-03-05 13:44:54 


In [2]:
library(bigstatsr)
library(rsvd)
library(here)
library(dplyr)

source(here("config.R"))

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Configuration

In [3]:
source_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_multiplier_imp_fixed")
output_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2_svd_k_div4")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

sample_sizes <- c(500, 1000, 2000, 4000, 8000, 16000, 32000)
n_seeds      <- 3
N_CORES      <- config$recount2$N_CORES

message("Source dir   : ", source_dir)
message("Output dir   : ", output_dir)
message("Sample sizes : ", paste(sample_sizes, collapse = ", "))
message("Seeds per size: ", n_seeds)

Source dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_multiplier_imp_fixed

Output dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_svd_k_div4

Sample sizes : 500, 1000, 2000, 4000, 8000, 16000, 32000

Seeds per size: 3



## Load preprocessed FBM

Reuses the cached preprocessed FBM created by `07_recount2_coverage_multiplier_imp_fixed.ipynb`.

In [4]:
preproc_fbm_rds <- file.path(source_dir, "FBMrecount2_cov_preproc_filtered.rds")

if (!file.exists(preproc_fbm_rds)) {
    stop("Preprocessed FBM not found. Run 07_recount2_coverage_multiplier_imp_fixed.ipynb first.")
}

recount2_fbm_filt <- readRDS(preproc_fbm_rds)
n_genes       <- nrow(recount2_fbm_filt)
n_samps_total <- ncol(recount2_fbm_filt)
message("Loaded preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")

Loaded preprocessed FBM: 6000 genes x 37032 samples



## Compute big_randomSVD and rsvd for all subsamples (K/4)

For each run, sample indices are read from `subsample_info.rds` in the source directory.

- `big_randomSVD`: runs directly on the FBM with `ind.col = sample_idx` and `k = SVD_K`.
- `rsvd`: submatrix is extracted from the FBM and passed to `rsvd::rsvd()` with `k = SVD_K`.
- `SVD_K = floor(min(n_samples − 1, n_genes − 1) / 4)`.

Skips any run where both `svd.rds` and `svd_rsvd.rds` already exist.

In [5]:
for (n_target in sample_sizes) {
    for (run_idx in seq_len(n_seeds)) {

        src_run_dir <- file.path(
            source_dir,
            paste0("c2cp_subsample_", n_target, "_seed_", run_idx)
        )

        if (!dir.exists(src_run_dir)) {
            message("Skipping (source not found): ", src_run_dir)
            next
        }

        out_run_dir <- file.path(
            output_dir,
            paste0("c2cp_subsample_", n_target, "_seed_", run_idx)
        )
        dir.create(out_run_dir, showWarnings = FALSE, recursive = TRUE)

        out_bigsvd_rds <- file.path(out_run_dir, "svd.rds")
        out_rsvd_rds   <- file.path(out_run_dir, "svd_rsvd.rds")

        if (file.exists(out_bigsvd_rds) && file.exists(out_rsvd_rds)) {
            message(sprintf("Already exists — skipping n=%d run=%d", n_target, run_idx))
            next
        }

        message(sprintf("\n── n=%d  run=%d ──────────────────────────────", n_target, run_idx))

        # ── Recover sample indices from source dir ────────────────────
        info       <- readRDS(file.path(src_run_dir, "subsample_info.rds"))
        sample_idx <- info$sample_idx
        n_samples  <- info$n_samples
        message("  n_samples : ", n_samples)

        # Copy subsample_info.rds and CLAMP_K.rds to new dir
        file.copy(file.path(src_run_dir, "subsample_info.rds"),
                  file.path(out_run_dir,  "subsample_info.rds"), overwrite = TRUE)
        file.copy(file.path(src_run_dir, "CLAMP_K.rds"),
                  file.path(out_run_dir,  "CLAMP_K.rds"), overwrite = TRUE)

        # ── Compute SVD_K = floor(K_full / 4) ────────────────────────
        SVD_K_full <- min(n_samples - 1L, n_genes - 1L)
        SVD_K      <- floor(SVD_K_full / 4L)
        message(sprintf("  SVD_K = floor(%d / 4) = %d", SVD_K_full, SVD_K))

        # ── 1. big_randomSVD ──────────────────────────────────────────
        if (!file.exists(out_bigsvd_rds)) {
            message(sprintf("  Computing big_randomSVD (k=%d)...", SVD_K))

            if (N_CORES > 1) {
                options(bigstatsr.check.parallel.blas = FALSE)
                blas_nproc <- getOption("default.nproc.blas")
                options(default.nproc.blas = NULL)
            }

            t_bigsvd <- system.time({
                svd_big <- big_randomSVD(
                    recount2_fbm_filt,
                    k       = SVD_K,
                    ind.col = sample_idx,
                    ncores  = N_CORES
                )
            })["elapsed"]

            if (N_CORES > 1) {
                options(bigstatsr.check.parallel.blas = TRUE)
                options(default.nproc.blas = blas_nproc)
            }

            message(sprintf("  big_randomSVD done in %.1fs", t_bigsvd))

            valid_idx    <- which(!is.nan(svd_big$d))
            svd_big$d    <- svd_big$d[valid_idx]
            svd_big$u    <- svd_big$u[, valid_idx, drop = FALSE]
            svd_big$v    <- svd_big$v[, valid_idx, drop = FALSE]

            saveRDS(svd_big, out_bigsvd_rds)
            message("  Saved: ", out_bigsvd_rds)
            rm(svd_big)
            gc()
        } else {
            message("  svd.rds already exists — skipping big_randomSVD")
        }

        # ── 2. rsvd ───────────────────────────────────────────────────
        if (!file.exists(out_rsvd_rds)) {
            message("  Extracting submatrix from FBM...")
            Y_sub <- recount2_fbm_filt[, sample_idx]

            message(sprintf("  Computing rsvd (k=%d)...", SVD_K))
            t_rsvd <- system.time({
                svd_rsvd <- rsvd::rsvd(Y_sub, k = SVD_K)
            })["elapsed"]
            message(sprintf("  rsvd done in %.1fs", t_rsvd))

            valid_idx        <- which(!is.nan(svd_rsvd$d))
            svd_rsvd$d       <- svd_rsvd$d[valid_idx]
            svd_rsvd$u       <- svd_rsvd$u[, valid_idx, drop = FALSE]
            svd_rsvd$v       <- svd_rsvd$v[, valid_idx, drop = FALSE]

            saveRDS(svd_rsvd, out_rsvd_rds)
            message("  Saved: ", out_rsvd_rds)
            rm(Y_sub, svd_rsvd)
            gc()
        } else {
            message("  svd_rsvd.rds already exists — skipping rsvd")
        }
    }
}

message("\nAll SVDs computed (K/4).")


── n=500  run=1 ──────────────────────────────

  n_samples : 500

  SVD_K = floor(499 / 4) = 124

  Computing big_randomSVD (k=124)...

  big_randomSVD done in 4.6s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_svd_k_div4/c2cp_subsample_500_seed_1/svd.rds

  Extracting submatrix from FBM...

  Computing rsvd (k=124)...

  rsvd done in 0.8s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_svd_k_div4/c2cp_subsample_500_seed_1/svd_rsvd.rds


── n=500  run=2 ──────────────────────────────

  n_samples : 500

  SVD_K = floor(499 / 4) = 124

  Computing big_randomSVD (k=124)...

  big_randomSVD done in 4.0s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_svd_k_div4/c2cp_subsample_500_seed_2/svd.rds

  Extracting submatrix from FBM...

  Computing rsvd (k=124)...

  rsvd done in 1.2s

  Saved: /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2_svd_k_div4/c2cp_subsample_500_seed_2/svd_rsvd.rds


── n=

In [6]:
end_time     <- Sys.time()
elapsed_time <- end_time - start_time
cat("Completed at:", format(end_time), "\n")
cat("Elapsed     :", format(elapsed_time), "\n")

Completed at: 2026-03-05 17:16:31 
Elapsed     : 3.527043 hours 
